### Objective: EDA on sample dataset
- perform EDA on AWS Web Crawl Data
- Apply different in-built and user-defined functions

In [1]:
from pyspark.sql import functions as F
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession

from pyspark.sql import types

import pandas as pd
import numpy as np

In [2]:
### Window functions
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number

#### Configuration
- Set the configuration with `local[*]`. It allows to use all the available `cores` of the local computer
- `Local Mode`: This is ideal for testing or smaller-scale data processing where we don't nee a large distributed cluster

In [3]:
config = SparkConf()\
    .setAppName('AWS_Crawl')\
    .setMaster('local[*]')

#### Spark Session

In [4]:
spark = SparkSession \
    .builder \
    .config(conf = config)\
    .getOrCreate()

#### Read file from web crawl

In [5]:
# Path to data source
path= "../../data_aws/cdx-00009.gz"

In [6]:
# df_web = spark.read\
#     .option('header', True)\
#     .csv("../data_sources/AWS_web_crawl/cdx-00000")

df_web = spark.read\
    .csv(path)

In [7]:
df_web.show(5)

+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+------------------+----+--------------------+
|_c0|                 _c1|                 _c2|                 _c3|             _c4|                 _c5|                _c6|                 _c7|                 _c8|                _c9|              _c10|_c11|                _c12|
+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+------------------+----+--------------------+
|biz|decenter)/event-d...| "mime": "text/html"| "mime-detected":...| "status": "200"| "digest": "I4JIP...| "length": "193243"| "offset": "79817...| "filename": "cra...| "charset": "UTF-8"| "languages": "jpn|eng"| "truncated": "le...|
|biz|decenter)/event-d...| "mime": "text/html"| "mime-detected":

### 1. EDA: Basic Functions

In [8]:
df_web.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)



In [9]:
%%time
df_web[["_c2"]].count()

CPU times: total: 15.6 ms
Wall time: 32.2 s


14916360

##### Get Schema of the `dataframe`

In [10]:
df_web_sample= df_web.sample(0.05)

In [11]:
%%time
df_web_sample.count()

CPU times: total: 0 ns
Wall time: 34.4 s


746027

In [12]:
df_web_sample.show(5)

+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+
|_c0|                 _c1|                 _c2|                 _c3|             _c4|                 _c5|                _c6|                 _c7|                 _c8|                _c9|                _c10|                _c11|                _c12|
+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+
|biz|decenter)/event-d...| "mime": "text/html"| "mime-detected":...| "status": "200"| "digest": "ZZSXH...| "length": "192829"| "offset": "79600...| "filename": "cra...| "charset": "UTF-8"|   "languages": "jpn|                eng"| "truncated": 

#### Describe the dataframe

In [13]:
df_web_sample.describe().show()

+-------+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+--------------------+
|summary|   _c0|                 _c1|                 _c2|                 _c3|                 _c4|                 _c5|                 _c6|                 _c7|                 _c8|               _c9|              _c10|                _c11|                _c12|
+-------+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+--------------------+--------------------+
|  count|746027|              746027|              746019|              745998|              745998|              746004|              746023|              744833|              744585|            722252|  

#### Check if `categorical` or `numerical` values

In [15]:
# df_web_sample._get_numeric_data().columns

### 2. EDA Level, Intermediate Functions
- More in-depth analysis
- EG: detecting null-values, grouping data, and creating basic aggregations

#### a. Check the column values by specific column name

In [16]:
df_web_sample.select("_c2").show(5)

+--------------------+
|                 _c2|
+--------------------+
| "mime": "text/html"|
| "mime": "text/html"|
| "mime": "text/html"|
| "mime": "text/html"|
| "mime": "text/html"|
+--------------------+
only showing top 5 rows



#### b. Check for `null` values
- Retrieve all the records that has `NULL` values by specific column

In [17]:
df_web_sample.filter(df_web_sample._c4.isNull()).show()

+---+--------------------+------+--------------------+----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|_c0|                 _c1|   _c2|                 _c3| _c4|                 _c5|                 _c6|                 _c7|                 _c8|                 _c9|                _c10|                _c11|                _c12|
+---+--------------------+------+--------------------+----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|biz|meanmachines)/sto...|  NULL|/form_key/llu1349...|NULL|/form_key/LLU1349...| "mime": "text/html"| "mime-detected":...|     "status": "302"| "digest": "3I42H...|     "length": "945"| "offset": "34642...| "filename": "cra...|
|biz|meanmachines)/sto...|  NULL|/form_key/fbgtebl...|NULL|/form_key/FbgTEbl...| "mime":

#### c. `groupBy()` and `agg()`
- Perform Aggregation on a single or mulitple columns

In [18]:
df_web_sample.groupBy("_c2").count().show()

+--------------------+-----+
|                 _c2|count|
+--------------------+-----+
|moi)/kupit-super-...|    1|
|o-cedar-commercia...|    1|
|i-shop)/pack-3.ht...|    1|
|help)/i4/i4design...|    1|
|abercrombie)/item...|    1|
|iccrea)/it/notizi...|    1|
|metaverseomni)/co...|    1|
|xn--cckp8i1dg9hu0...|    1|
|ads)/adclick.php?...|    1|
|en)/m/2yygs4e 202...|    1|
|sales)/919636864-...|    1|
|blog)/archives/66...|    1|
|business)/list/me...|    1|
|m)/free-porn-vide...|    1|
|kadenka)/search?g...|    1|
|demo-magento)/cat...|    1|
|blogbr)/arredores...|    1|
|contour-l5)/ 2023...|    1|
|ats)/posts/2023-1...|    1|
|labs)/index.php?d...|    1|
+--------------------+-----+
only showing top 20 rows



#### d. Select Expression on column
- on selected column perform transformations

In [19]:
df_web_sample.selectExpr("_c8 as STATUS").show()

+--------------------+
|              STATUS|
+--------------------+
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
| "filename": "cra...|
+--------------------+
only showing top 20 rows



#### e. Cross-Tab
- This in-built function allows to compute a contingency table of selected categorical variable columns
- EG:  df_web_sample.crosstab("column_1", "column_2")

In [21]:
# df_web_sample.crosstab("_c2", "_c5")

### 3. Advanced Level EDA Functions

In [22]:
df_web_sample.show(3)

+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+
|_c0|                 _c1|                 _c2|                 _c3|             _c4|                 _c5|                _c6|                 _c7|                 _c8|                _c9|               _c10|                _c11|                _c12|
+---+--------------------+--------------------+--------------------+----------------+--------------------+-------------------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+
|biz|decenter)/event-d...| "mime": "text/html"| "mime-detected":...| "status": "200"| "digest": "ZZSXH...| "length": "192829"| "offset": "79600...| "filename": "cra...| "charset": "UTF-8"|  "languages": "jpn|                eng"| "truncated": "le.

#### a. window functions
- Provides advanced analytical functions like: row number, rank, moving averages

In [23]:
window_spec= Window.orderBy("_c6")
df_web_sample.withColumn("Rank_by_type", rank().over(window_spec)).show()

+----+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------+
| _c0|               _c1|                 _c2|                 _c3|                 _c4|             _c5|                 _c6|                 _c7|                 _c8|                 _c9|                _c10|                _c11|              _c12|Rank_by_type|
+----+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------+
| biz|wyborcza)/waluty/0|              111140|             9252672|                NULL|            NULL|                NULL|                NULL|gbp.html?disabler...|              111140|             925267

#### b. Statistics:
- Perform statisitcal analysis on the selected `numerical` columns

In [24]:
# print(df_web_sample.stat.corr("_c1", "_c6"))

IllegalArgumentException: requirement failed: Currently correlation calculation for columns with dataType string not supported.

#### c. Frequency of Items
- Returns the frequent items by selected column with set threshold

In [ ]:
df_web_sample.stat.freqItmes(["_c8"], 0.3).show()